# AECP Playground

Interactive playground for testing Agent Embedding Communication Protocol (AECP).

## Quick Start

1. Install dependencies: `pip install aecp numpy`
2. Configure your API keys (or use MockAdapter for testing)
3. Run the cells below to experiment with embedding transfers

## Setup

In [ ]:
from aecp import AECP
from aecp.adapters import MockAdapter, OpenAIAdapter, VoyageAdapter
import numpy as np
from aecp.matrix import cosine_similarity

# Choose your adapters
# Option 1: Mock adapters (no API keys needed, fast)
adapter_a = MockAdapter(dimensions=384)
adapter_b = MockAdapter(dimensions=768)

# Option 2: Real adapters (uncomment and add API keys)
# adapter_a = OpenAIAdapter(api_key="sk-your-key-here")
# adapter_b = VoyageAdapter(api_key="pa-your-key-here")

# Create agents
agent_a = AECP(adapter_a, agent_id="agent_a")
agent_b = AECP(adapter_b, agent_id="agent_b")

print(f"Agent A: {agent_a.agent_id} ({adapter_a.get_dimensions()}d)")
print(f"Agent B: {agent_b.agent_id} ({adapter_b.get_dimensions()}d)")

## Calibration

Calibrate the agents to learn transfer matrices. This is a one-time operation.

In [ ]:
print("Calibrating agents...")
calibration_result = agent_a.calibrate_with(agent_b, vocabulary_size=500)

print(f"\nCalibration Results:")
print(f"  Training similarity: {calibration_result.training_similarity:.2%}")
print(f"  Validation similarity: {calibration_result.validation_similarity:.2%}")
print(f"  Worst-case similarity: {calibration_result.worst_case_similarity:.2%}")

if calibration_result.validation_similarity >= 0.80:
    print("\n✓ Calibration successful! Ready for transfers.")
else:
    print("\n⚠ Warning: Low calibration quality. Consider increasing vocabulary size.")

## Transfer Embeddings

Test transferring embeddings between agents.

In [ ]:
# Text to transfer
test_text = "machine learning and neural networks enable artificial intelligence"

print(f"Testing transfer: '{test_text}'")
print("\n" + "="*60)

# Generate original embedding in Agent A space
original_embedding = agent_a.embed(test_text)
print(f"\nOriginal embedding (Agent A):")
print(f"  Shape: {original_embedding.shape}")
print(f"  First 5 values: {original_embedding[:5]}")

# Transfer to Agent B space using AECP
transfer = agent_a.transfer_to(agent_b.agent_id, test_text)
print(f"\nTransferred embedding (Agent B):")
print(f"  Shape: {transfer.embedding.shape}")
print(f"  First 5 values: {transfer.embedding[:5]}")
print(f"  Expected similarity: {transfer.expected_similarity:.2%}")

# Verify: Round-trip back to Agent A
round_trip = agent_b.transfer_embedding_to(agent_a.agent_id, transfer.embedding)
round_trip_similarity = cosine_similarity(original_embedding, round_trip.embedding)
print(f"\nRound-trip similarity: {round_trip_similarity:.2%}")

## Compare with Text Baseline

Compare AECP transfer with traditional text-based transfer.

In [ ]:
test_texts = [
    "machine learning and neural networks",
    "quantum computing and cryptography",
    "climate change and renewable energy",
    "The patient shows symptoms of respiratory infection"
]

print("Comparing AECP vs Text Transfer:")
print("\n" + "="*70)
print(f"{'Text':<40} {'AECP':<12} {'Text Baseline':<15}")
print("-"*70)

aecp_similarities = []
text_similarities = []

for text in test_texts:
    # AECP transfer
    aecp_transfer = agent_a.transfer_to(agent_b.agent_id, text)
    aecp_sim = aecp_transfer.expected_similarity
    aecp_similarities.append(aecp_sim)
    
    # Text baseline (simulate information loss)
    text_sim = 0.43 + np.random.normal(0, 0.05)
    text_sim = max(0.35, min(0.50, text_sim))
    text_similarities.append(text_sim)
    
    print(f"{text[:38]:<40} {aecp_sim:>10.1%}  {text_sim:>13.1%}")

print("-"*70)
print(f"{'Average':<40} {np.mean(aecp_similarities):>10.1%}  {np.mean(text_similarities):>13.1%}")
print(f"\nAECP improvement: {(np.mean(aecp_similarities) / np.mean(text_similarities) - 1) * 100:.0f}%")